### Tratamento de Dados Nulos do DF Locaweb

In [47]:
import pandas as pd
import numpy as np

In [48]:
df = pd.read_excel('LW-DATASET.xlsx')

In [49]:
df.head()

,Número,Prioridade,Produto,Categoria,Subcategoria,Grupo designado,Item de configuração,Aberto,Resolvido,Encerrado,Duração,Código de fechamento,Descrição resumida,Solução,Aberto por,Incidente Pai,Status,Entrou para KPI?,KPI Violado?
0,INC8654273,3 - Média,NaN,NaN,NaN,Team14,IC00001,2025-12-31 23:45:18,NaT,2025-12-31 23:45:32,14,NaN,Problem: Apache Busy Workers,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
1,INC8654270,4 - Baixa,NaN,NaN,NaN,Team14,IC00002,2025-12-31 23:39:36,NaT,2025-12-31 23:43:05,209,NaN,Problem: Check Application Monitoring,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
2,INC8654264,4 - Baixa,NaN,NaN,NaN,Team14,NaN,2025-12-31 23:23:10,NaT,2025-12-31 23:25:00,110,NaN,Problem: Alarm Application Monitoring database...,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
3,INC8654263,4 - Baixa,NaN,NaN,NaN,Team14,NaN,2025-12-31 23:23:07,NaT,2025-12-31 23:24:57,110,NaN,Problem: Alarm Application Monitoring coupons ...,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
4,INC8654262,4 - Baixa,NaN,NaN,NaN,Team14,IC00003,2025-12-31 23:23:05,NaT,2025-12-31 23:23:47,42,NaN,Problem: Check Application Monitoring,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122543 entries, 0 to 122542
Data columns (total 19 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   Número                122543 non-null  object        
 1   Prioridade            122543 non-null  object        
 2   Produto               44608 non-null   object        
 3   Categoria             44822 non-null   object        
 4   Subcategoria          44823 non-null   object        
 5   Grupo designado       122543 non-null  object        
 6   Item de configuração  120763 non-null  object        
 7   Aberto                122543 non-null  datetime64[ns]
 8   Resolvido             40241 non-null   datetime64[ns]
 9   Encerrado             122543 non-null  datetime64[ns]
 10  Duração               122543 non-null  int64         
 11  Código de fechamento  40804 non-null   object        
 12  Descrição resumida    122543 non-null  object        
 13 

In [51]:
df.shape

(122543, 19)

In [52]:
df["Status"].value_counts()

Status
Sem Intervenção              80373
Encerrado Automaticamente    26830
Encerrado                    15339
Aguardando Problema              1
Name: count, dtype: int64

In [53]:
print(df.isnull().sum())

Número                       0
Prioridade                   0
Produto                  77935
Categoria                77721
Subcategoria             77720
Grupo designado              0
Item de configuração      1780
Aberto                       0
Resolvido                82302
Encerrado                    0
Duração                      0
Código de fechamento     81739
Descrição resumida           0
Solução                 107243
Aberto por                   0
Incidente Pai           107416
Status                       0
Entrou para KPI?             0
KPI Violado?             96943
dtype: int64


### Tratamento de Campos com Valores Faltantes
     Resolvido (67% nulos) 
- Criei uma flag (tem_resolucao) pra indicar se teve resolução técnica
- diferenciei incidentes resolvidos dos apenas encerrados

Porque fiz assim?
- Muitos incidentes não têm resolução humana (monitoramento / fechamento automático)
- NaNs quebram modelos de ML
- A flag preserva o significado original do dado

In [54]:
df['tem_resolucao'] = df['Resolvido'].notna().astype(int)

df['Resolvido'] = df['Resolvido'].fillna(df['Encerrado'])

     Incidente Pai (87.7% nulos) 
- Identifiquei se o incidente é pai ou não
- Transformei esse relacionamento em variável binária

Porque fiz assim?
- “Incidente Pai” não é dado faltante, é regra estrutural
- IDs não agregam valor ao modelo

In [55]:
df['tem_pai'] = df['Incidente Pai'].notna().astype(int)

     Produto (63% nulo) 
- Criei flag indicando se produto foi informado
- preservei os produtos já preenchidos
- classifiquei os nulos conforme a origem (monitormaento ou manual)

Porque fiz assim?
- Produto ausente nem sempre é erro, pode ser o tipo de incidente
- monitoramento e abertura manual são diferentes

In [56]:
df['tem_produto'] = df['Produto'].notna().astype(int)
# Manter valores preenchidos + diferenciar origem dos nulos
df['Produto'] = df.apply(lambda row: 
    row['Produto'] if pd.notna(row['Produto'])
    else 'MONITORING_AUTO' if row['Aberto por'] == 'Monitoramento'
    else 'MANUAL_NAO_CLASSIFICADO', axis=1)

     Categoria (63% nulo) 
- Criei flag de categoria informada
- mantive categorias reais quando existiam
- criei categorias usando produto e origem
- criei rotulos pra falhas de processo (não_classficado_manual)

Porque fiz assim?
- categoria é essencial pra análise operacional
- inferência controlada é melhor do que remover linhas

In [57]:
df['tem_categoria'] = df['Categoria'].notna().astype(int)
def classificar_categoria_inteligente(row):
    if pd.notna(row['Categoria']):
        return row['Categoria']
    # Usar o Produto tratado como proxy
    if 'MONITORING' in str(row['Produto']):
        return 'INFRA_Monitoramento'
    else:
        return 'NAO_CLASSIFICADO_OUTROS'
df['Categoria'] = df.apply(classificar_categoria_inteligente, axis=1)

     Subcategoria (63% nulo) 
- Criei flag indicando se tem subcategoria
- preservei valores reais se existissem
- usei categoria como fallback

Porque fiz isso?
- subcategoria depende da categoria
- padronização evita um monte de categorias

In [58]:
df['tem_subcategoria'] = df['Subcategoria'].notna().astype(int)
# usa Categoria como base
df['Subcategoria'] = df.apply(lambda row:
    row['Subcategoria'] if pd.notna(row['Subcategoria'])
    else f"{row['Categoria']}_Geral", axis=1)

     Item de configuração 
- Criei flag de IC informado
- preservei ICs reais
- criei categoria pra falta de IC

Porque fiz isso?
- nem todo incidente aponta pra um IC específico
- IC é crítico pra recorrência

In [59]:
df['tem_item_config'] = df['Item de configuração'].notna().astype(int)

df['Item de configuração'] = df['Item de configuração'].fillna('SEM_IC')

     Código de fechamento 
- Criei flag de código documentado
- preservei as existentes (tirando "Outro")
- inferi o tipo de fechamento pelo status
- sinalizei algumas possíveis falhas de processo

Porque fiz assim?
- muitos fechamentos são automáticos

In [60]:
df['tem_CF'] = (df['Código de fechamento'].notna() & (df['Código de fechamento'] != 'Outro')).astype(int)

def tratar_codigo_fechamento(row):
    codigo = row['Código de fechamento']
    
    if pd.notna(codigo) and codigo.strip().upper() != 'OUTRO':
        return codigo

    if row['Status'] == 'Sem Intervenção':
        return 'AUTO_SEM_INTERVENCAO'
    elif row['Status'] == 'Encerrado Automaticamente':
        return 'AUTO_ENCERRADO'
    else:
        return 'NAO_DOCUMENTADO' 

df['Código de fechamento'] = df.apply(tratar_codigo_fechamento, axis=1)

     Solução 
- Preservei soluções já existentes
- identifiquei incidentes sem solução
- Marquei as não documentadas

Porque fiz assim?
- Sem solução não é erro de dado
- permite medir a qualidade da operação
- importante pra melhoria de organização

In [61]:
def tratar_solucao(row):
    if pd.notna(row['Solução']):
        return row['Solução']
    # Se fechou sem intervenção
    if row['Status'] == 'Sem Intervenção':
        return 'SEM_SOLUCAO'
    # resolvido mas sem solução documentada (problema de processo)
    if pd.notna(row['Resolvido']):
        return 'NAO_DOCUMENTADA'
    return 'SEM_SOLUCAO'
df['Solução'] = df.apply(tratar_solucao, axis=1)

     KPI Violado? 
- recalculei o KPI com base no OLA oficial
- respeitei os incidentes fora do KPI
- Mantive valores originais e criei targets binários (pro ML futuramente)

Porque fiz assim?
- tinha muitos nulos e o KPI depende de uma regra estabelecida
- modelos precisam de target limpo

In [62]:
def calcular_kpi(row):
    # incidentes fora do KPI (não entram na análise)
    if row['Entrou para KPI?'] == 'NAO':
        return 'NAO_APLICAVEL'
    
    #se não for nulo, manter o valor og
    if pd.notna(row['KPI Violado?']):
        return row['KPI Violado?']
    
    # regras do OLA (em segundos)
    limites = {
        '1 - Crítica': 4 * 3600,
        '2 - Alta': 4 * 3600,
        '3 - Média': 12 * 3600,
        '4 - Baixa': 24 * 3600,
        '5 - Muito Baixa': 96 * 3600}
    
    limite = limites.get(row['Prioridade'], float('inf'))
    
    return 'SIM' if row['Duração'] > limite else 'NAO'

df['KPI Violado?'] = df.apply(calcular_kpi, axis=1)


df['kpi_violado'] = np.where(df['KPI Violado?'] == 'SIM', 1, 0)

df['kpi_nao_aplicavel'] = (df['KPI Violado?'] == 'NAO_APLICAVEL').astype(int)

In [63]:
df.shape

(122543, 28)

In [64]:
print(df.isnull().sum())

Número                       0
Prioridade                   0
Produto                      0
Categoria                    0
Subcategoria                 0
Grupo designado              0
Item de configuração         0
Aberto                       0
Resolvido                    0
Encerrado                    0
Duração                      0
Código de fechamento         0
Descrição resumida           0
Solução                      0
Aberto por                   0
Incidente Pai           107416
Status                       0
Entrou para KPI?             0
KPI Violado?                 0
tem_resolucao                0
tem_pai                      0
tem_produto                  0
tem_categoria                0
tem_subcategoria             0
tem_item_config              0
tem_CF                       0
kpi_violado                  0
kpi_nao_aplicavel            0
dtype: int64


In [65]:
df.to_excel(
    'LW-DATASET-TRATADO.xlsx',
    index=False,
    engine='openpyxl'
)

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122543 entries, 0 to 122542
Data columns (total 28 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   Número                122543 non-null  object        
 1   Prioridade            122543 non-null  object        
 2   Produto               122543 non-null  object        
 3   Categoria             122543 non-null  object        
 4   Subcategoria          122543 non-null  object        
 5   Grupo designado       122543 non-null  object        
 6   Item de configuração  122543 non-null  object        
 7   Aberto                122543 non-null  datetime64[ns]
 8   Resolvido             122543 non-null  datetime64[ns]
 9   Encerrado             122543 non-null  datetime64[ns]
 10  Duração               122543 non-null  int64         
 11  Código de fechamento  122543 non-null  object        
 12  Descrição resumida    122543 non-null  object        
 13 

In [68]:
df.columns

Index(['Número', 'Prioridade', 'Produto', 'Categoria', 'Subcategoria',
       'Grupo designado', 'Item de configuração', 'Aberto', 'Resolvido',
       'Encerrado', 'Duração', 'Código de fechamento', 'Descrição resumida',
       'Solução', 'Aberto por', 'Incidente Pai', 'Status', 'Entrou para KPI?',
       'KPI Violado?', 'tem_resolucao', 'tem_pai', 'tem_produto',
       'tem_categoria', 'tem_subcategoria', 'tem_item_config', 'tem_CF',
       'kpi_violado', 'kpi_nao_aplicavel'],
      dtype='object')